# 강의계획서 자동화 (Colab)

강사용 입력 엑셀을 마스터 템플릿 PPTX 초안으로 자동 변환합니다. **여러 파일을 올리면 하나의 통합 PPTX**로 생성합니다.
위에서부터 **순서대로 실행**하세요.

**사전 준비 (한 번만)**: `lecture-plan-automation` 폴더(`src`, `config`, `templates`, `teacher_photos` 포함)를 **내 드라이브(MyDrive)** 에 업로드해 두세요.
→ drive.google.com 에서 폴더째로 드래그. 드라이브에 두면 런타임이 끊겨도 사라지지 않습니다.

자동 적용 기능: 수강료 자동 계산(정규반=월 단위 / 특강·썸머=전체 합계) · 과목별 색 테두리 · 강사 사진 · 박스 자동 확장.
입력 엑셀은 **3번 셀에서 그때그때 업로드**(여러 개 동시 가능)합니다.


## 1. 패키지 설치


In [ ]:
!pip -q install pandas openpyxl python-pptx

## 2. 드라이브 연결 + 프로젝트 폴더 찾기
팝업이 뜨면 계정 인증을 진행하세요. `src 존재: True`, `템플릿 존재: True`로 나오면 정상입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob

PROJECT_DIR = '/content/drive/MyDrive/lecture-plan-automation'  # 드라이브 내 폴더 경로 (다르면 수정)

# 경로가 틀려도 내 드라이브에서 프로젝트를 자동 탐색(1~2단계 깊이)
if not os.path.exists(os.path.join(PROJECT_DIR, 'src', 'pipeline.py')):
    cand = (glob.glob('/content/drive/MyDrive/*/src/pipeline.py')
            + glob.glob('/content/drive/MyDrive/*/*/src/pipeline.py'))
    if cand:
        PROJECT_DIR = os.path.dirname(os.path.dirname(cand[0]))

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)   # 'src'/'config' 패키지 import 가능하게

print('작업 폴더:', os.getcwd())
print('src 존재:', os.path.exists('src/pipeline.py'))
print('템플릿 존재:', os.path.exists('templates/강의계획서_마스터템플릿.pptx'))

## 3. 강사 입력 엑셀 업로드 (여러 개 동시 가능)
파일 선택창에서 **Ctrl(또는 Shift) 클릭으로 여러 개를 한꺼번에** 선택하세요. 올린 엑셀들의 모든 강좌가 **하나의 통합 PPTX**로 합쳐집니다.


In [ ]:
from google.colab import files
import shutil, unicodedata

def nfc(s):
    return unicodedata.normalize('NFC', s)   # 한글 자모분리(NFD) → 정상 결합(NFC)

UPLOAD_DIR = '_uploads'
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)
os.makedirs(UPLOAD_DIR)

uploaded = files.upload()   # 엑셀(.xlsx) 여러 개 선택 가능
n = 0
for fn, data in uploaded.items():
    if fn.lower().endswith('.xlsx'):
        with open(os.path.join(UPLOAD_DIR, nfc(fn)), 'wb') as f:
            f.write(data)
        n += 1
print(f'업로드된 엑셀 {n}개 → 하나의 통합 PPTX로 처리합니다.')

## 4. 대상 월 (정규반 월별 계획서)
`TARGET_MONTH`에 월 숫자를 넣으면 **정규반 강좌만** 그 달 진도로 잘라 회차·수강기간·수강료를 그 달 기준으로 재계산합니다. 전체 기간 계획서는 `None`. (`input()` 대신 변수라 셀 재실행에 안전)

In [ ]:
TARGET_MONTH = 7   # 생성할 월(정규반). 전체 기간 계획서는 None.

## 5. 실행 (통합 PPTX 생성)
올린 엑셀들의 모든 강좌를 한 PPTX에 슬라이드로 모읍니다. 수강료·색 테두리·사진·박스 확장은 자동 적용됩니다.

In [ ]:
from src.pipeline import run_pipeline

result = run_pipeline(
    input_path=UPLOAD_DIR,
    output_dir='output',
    base_year=2026,
    make_pptx=True,
    target_month=TARGET_MONTH,   # None이면 전체 기간 / 숫자면 그 달 정규반
)
print('통합 강좌 수:', result['lecture_count'])
result

## 6. (선택) 결과 미리보기
통합 PPTX의 모든 슬라이드를 이미지로 확인합니다. 처음 1회 LibreOffice 설치에 1~2분 걸립니다.
강좌가 많으면 시간이 걸리니, 빠르게 보려면 다음 셀의 `MAX_PAGES`를 줄이세요.

In [ ]:
MAX_PAGES = 8   # 미리보기 최대 슬라이드 수(전체는 6번 zip으로 받으세요)

!apt-get -qq install -y libreoffice poppler-utils >/dev/null
!pip -q install pdf2image
from pdf2image import convert_from_path

pptx = result['pptx_path']
!libreoffice --headless --convert-to pdf --outdir /tmp "{pptx}" >/dev/null
pdf = sorted(glob.glob('/tmp/*.pdf'), key=os.path.getmtime)[-1]
for img in convert_from_path(pdf, dpi=110)[:MAX_PAGES]:
    display(img)

## 7. 결과 다운로드 (zip 한 개)
통합 PPTX·정규화 데이터·검증 리포트를 zip 하나로 묶어 내려받습니다. 한글 파일명은 NFC로 정규화해 Windows에서도 안 깨집니다.

In [ ]:
import shutil, unicodedata, os
from google.colab import files

def nfc(s):
    return unicodedata.normalize('NFC', s)

# zip·다운로드는 드라이브가 아니라 로컬(/content)에서 처리합니다.
# (드라이브는 네트워크 마운트라 압축/다운로드가 매우 느림)
BUNDLE_DIR = '/content/강의계획서_결과'
if os.path.exists(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)
os.makedirs(BUNDLE_DIR)

for key in ['pptx_path', 'normalized_xlsx', 'normalized_json', 'validation_report']:
    p = result.get(key)
    if p and os.path.exists(p):
        shutil.copy2(p, os.path.join(BUNDLE_DIR, nfc(os.path.basename(p))))

ZIP_PATH = '/content/강의계획서_결과.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive('/content/강의계획서_결과', 'zip', '/content', '강의계획서_결과')
print('압축 완료: %s (%.1f MB)' % (ZIP_PATH, os.path.getsize(ZIP_PATH)/1024/1024))
files.download(ZIP_PATH)
